# Kgent Agent Loop 逐步拆解（DeepSeek）

本 Notebook 用**同一条代码路径**（`run_agent_stream`）跑一轮对话，并在每个 **checkpoint** 打印当时 session 里有什么、发给 **DeepSeek** 的请求长什么样。

## 准备

```powershell
cd D:\Kgent
pip install -e ".[dev,notebook]"
```

项目根 `.env` 需配置 DeepSeek（OpenAI 兼容接口）：

```env
KGENT_PROVIDER=openai
KGENT_MODEL=deepseek-chat
KGENT_API_KEY=你的_deepseek_key
KGENT_BASE_URL=https://api.deepseek.com
```

启动：

```powershell
jupyter notebook notebooks/kgent_loop_walkthrough.ipynb
```

## 你会看到什么

| 阶段 | 含义 |
|------|------|
| `after_user_append` | 用户话写进 session |
| `before_model_call` | **messages[] + tools[]** 即将发给 DeepSeek |
| `after_model` | 模型返回（文本或 tool_calls）写入 session |
| `after_tool` | Python 执行工具，**tool_result** 写回 session |
| `complete` | 模型不再调工具，给出最终答案 |


In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from nb_helpers import setup_python_path, ensure_deepseek_ready

REPO_ROOT = setup_python_path()
print("repo_root:", REPO_ROOT)

settings = ensure_deepseek_ready()
print("project_root:", settings.project_root)


## 1. 工具注册表 → `tool_schemas[]`

Loop 启动时 `build_tools()` 得到 Python 工具实例，再投影成发给 LLM 的 schema（**不含** `risk_level`）。

这些 schema **不会**写进 system prompt 字符串，而是在 `call_model` 时作为 API 的 `tools[]` 参数并列发送。


In [ ]:
from nb_helpers import show_tool_registry

show_tool_registry()


## 2. 跑一轮 Agent Loop（DeepSeek + function calling）

与 WebSocket 相同：`plan_before_act=False`，真实调用 `https://api.deepseek.com`。

改 `USER_INPUT` 做实验。


In [ ]:
from nb_helpers import run_walkthrough

USER_INPUT = "计算 8+8"
# DeepSeek 走 OpenAI 兼容 client；配置来自 .env
PROVIDER = "openai"

run = await run_walkthrough(
    USER_INPUT,
    session_id="notebook-demo",
    provider=PROVIDER,
    plan_before_act=False,
    reset_session=True,
)

print("model:", run.model)
print("checkpoints:", len(run.checkpoints))
print("FINAL ANSWER:", run.answer)
print("session message_count:", run.message_count)


## 3. 每个 checkpoint 的产出

重点看 **`before_model_call`**：此时会预览 OpenAI/DeepSeek 请求体（`messages` + `tools`）。

**`after_model`**：看 assistant 是否带 `tool_use`。

**`after_tool`**：看 `tool_result` 如何作为 user 侧观察写回 session。


In [ ]:
from nb_helpers import display_checkpoint

for index, event in enumerate(run.checkpoints, start=1):
    display_checkpoint(event, index=index, model=run.model)


## 4. 多轮 session（可选）

同一 `session_id` 再跑一条消息，观察 **user context** 如何累积（system 仍在 `[0]`）。


In [ ]:
run2 = await run_walkthrough(
    "刚才算的结果是多少？",
    session_id="notebook-demo",
    provider=PROVIDER,
    reset_session=False,
)

from IPython.display import Markdown, display

display(Markdown(f"**第二轮最终答案:** {run2.answer}"))
for index, event in enumerate(run2.checkpoints, start=1):
    if event.payload.get("checkpoint") == "before_model_call":
        display_checkpoint(event, index=index, model=run2.model)


## 5. 对照：Debug CLI 的 plan 模式

Debug CLI 默认 `plan_before_act=True`：每轮先 **无 tools** 的 plan call，再带 tools 的 act call。

在 notebook 里打开即可对比 checkpoint 名称差异（`before_plan_call` / `after_plan` / `before_model_call`）。


In [ ]:
run_plan = await run_walkthrough(
    "计算 12 * 8 + 6",
    session_id="notebook-plan",
    provider=PROVIDER,
    plan_before_act=True,
    reset_session=True,
)

print("checkpoints:", [e.payload["checkpoint"] for e in run_plan.checkpoints])
print("FINAL:", run_plan.answer)
